> **Updated November 2026.** Amazon Titan Text, Anthropic Claude 3.x and Stable Diffusion XL reached end of life on Amazon Bedrock (see [Model lifecycle](https://docs.aws.amazon.com/bedrock/latest/userguide/model-lifecycle.html)). This notebook now uses [Amazon Nova 2 Lite](https://docs.aws.amazon.com/bedrock/latest/userguide/model-card-amazon-nova-2-lite.html) through the [Converse API](https://docs.aws.amazon.com/bedrock/latest/userguide/conversation-inference-call.html) and [Claude Haiku 5.5](https://docs.aws.amazon.com/bedrock/latest/userguide/model-card-anthropic-claude-haiku-5-5.html) / [Claude Sonnet 5.5](https://docs.aws.amazon.com/bedrock/latest/userguide/model-card-anthropic-claude-sonnet-5-5.html) through cross-Region inference profiles. Claude 4.7+ no longer accepts `temperature`, `top_p` or `top_k` ([Anthropic docs](https://platform.claude.com/docs/en/models/haiku-5-5/overview)), so those parameters were removed from the Claude requests.

# Entity Extraction with Claude

> *This notebook should work well with the **`Python 3`** kernel in SageMaker Studio*

### Context
Entity extraction is an NLP technique that allows us to automatically extract specific data from naturally written text, such as news, emails, books, etc.
That data can then later be saved to a database, used for lookup or any other type of processing.

Classic entity extraction programs usually limit you to pre-defined classes, such as name, address, price, etc. or require you to provide many examples of types of entities you are interested in.
By using a LLM for entity extraction in most cases you are only required to specify what you need to extract in natural language. This gives you flexibility and accuracy in your queries while saving time by removing necessity of data labeling.

In addition, LLM entity extraction can be used to help you assemble a dataset to later create a customised solution for your use case, such as [Amazon Comprehend custom entity](https://docs.aws.amazon.com/comprehend/latest/dg/custom-entity-recognition.html) recognition.

## Setup


In [1]:
%pip install -U langchain-aws beautifulsoup4

In [2]:
import json
import os
import sys

import boto3
import botocore

boto3_bedrock = boto3.client('bedrock-runtime')

## Configure langchain

We begin with instantiating the LLM. Here we are using Anthropic Claude Haiku 5.5 for text generation, through `ChatBedrockConverse`, the class LangChain recommends for Bedrock ([LangChain docs](https://docs.langchain.com/oss/python/integrations/chat/bedrock)). Claude 4.7+ does not accept `temperature` ([Anthropic docs](https://platform.claude.com/docs/en/models/haiku-5-5/overview)), so it is not set. Claude 5.5 has [adaptive thinking](https://platform.claude.com/docs/en/build-with-claude/thinking) on by default: the response can start with a `thinking` block (Invoke) or a `reasoningContent` block (Converse) before the `text` block, so the code below reads `AIMessage.text` (the concatenated text blocks, see the [LangChain messages guide](https://docs.langchain.com/oss/python/langchain/models#stream)) instead of `.content`. These demos ask for short answers with a small token budget, so [adaptive thinking](https://platform.claude.com/docs/en/build-with-claude/thinking) is turned off explicitly with `additional_model_request_fields={"thinking": {"type": "disabled"}}`; otherwise the model can spend the whole `max_tokens` budget thinking and return no text.

Note: It is possible to choose other models available with Bedrock. For example, you can replace the `model_id` as follows to change the model to Amazon Nova 2 Lite. Make sure your account has access to the model you want to try out before trying this!

`llm = ChatBedrockConverse(model_id="us.amazon.nova-2-lite-v1:0", max_tokens=200)` (Nova does not take the `thinking` field).

In [3]:
from langchain_aws import ChatBedrockConverse

llm = ChatBedrockConverse(
    model_id="global.anthropic.claude-haiku-5-5",
    max_tokens=200,
    additional_model_request_fields={"thinking": {"type": "disabled"}},
)

## Entity Extraction
Now that we have our LLM initialised, we can start extracting entities.

For this exercise we will pretend to be an online bookstore that receives questions and orders by email.
Our task would be to extract relevant information from the email to process the order.

Let's begin by taking a look at the sample email:

In [4]:
from pathlib import Path

emails_dir = Path(".") / "emails"
with open(emails_dir / "00_treasure_island.txt") as f:
    book_question_email = f.read()

print(book_question_email)

Dear Sir or Madam,

I would like to order Treasure Island, do you have it in stock?

Also, is it possible to pay by cheque?

Yours sincerely,
John Smith


### Basic approach

For basic cases we can directly ask the model to return the result.
Let's try extracting the name of the book.

In [5]:
query = f"""
Given the email inside triple-backticks, please read it and analyse the contents.
If a name of a book is mentioned, return it, otherwise return nothing.

Email: ```
{book_question_email}
```

"""

messages = [
    (
        "system",
        "You are a helpful assistant that processes orders from a bookstore.",
    ),
    ("human", query),
]

In [6]:
result = llm.invoke(messages)
print(result.text)

Treasure Island


### Model specific prompts

While basic approach works, to achieve best results we recommend to customise your prompts for the particular model you will be using.
In this example we are using Claude Haiku 5.5, [prompt guide for which can be found here](https://platform.claude.com/docs/en/build-with-claude/prompt-engineering/overview).

Here is the a more optimised prompt for Claude.

In [7]:
prompt = """

Given the email provided, please read it and analyse the contents.
If a name of a book is mentioned, return it.
If no name is mentioned, return empty string.
The email will be given between <email></email> XML tags.

<email>
{email}
</email>

Return the name of the book between <book></book> XML tags.

"""

In [8]:
query = prompt.format(email=book_question_email)
messages = [
    (
        "system",
        "You are a helpful assistant that processes orders from a bookstore.",
    ),
    ("human", query),
]
result = llm.invoke(messages).text
print(result)

<book>Treasure Island</book>


To extract results easier, we can use a helper function:

In [9]:
from bs4 import BeautifulSoup

def extract_by_tag(response: str, tag: str, extract_all=False) -> str | list[str] | None:
    soup = BeautifulSoup(response)
    results = soup.find_all(tag)
    if not results:
        return
        
    texts = [res.get_text() for res in results]
    if extract_all:
        return texts
    return texts[-1]

In [10]:
extract_by_tag(result, "book")

'Treasure Island'

We can check that our model doesn't return arbitrary results when no appropriate information is given (also know as 'hallucination'), by running our prompt on other emails.

In [11]:
with open(emails_dir / "01_return.txt") as f:
    return_email = f.read()

print(return_email)

I didn't like the last book I ordered and would like to return it.


In [12]:
query = prompt.format(email=return_email)
messages = [
    (
        "system",
        "You are a helpful assistant that processes orders from a bookstore.",
    ),
    ("human", query),
]
result = llm.invoke(query).text
print(result)

<book></book>


Using tags also allows us to extract multiple pieces of information at the same time and makes extraction much easier.
In the following prompt we will extract not just the book name, but any questions, requests and customer name.

In [13]:
prompt = """

Human: Given email provided , please read it and analyse the contents.

Please extract the following information from the email:
- Any questions the customer is asking, return it inside <questions></questions> XML tags.
- The customer full name, return it inside <name></name> XML tags.
- Any book names the customer mentions, return it inside <books></books> XML tags.

If a particular bit of information is not present, return an empty string.
Make sure that each question can be understoon by itself, incorporate context if requred.
Each returned question should be concise, remove extra information if possible.
The email will be given between <email></email> XML tags.

<email>
{email}
</email>

Return each question inside <question></question> XML tags.
Return the name of each book inside <book></book> XML tags.

Assistant:"""

In [14]:
query = prompt.format(email=book_question_email)
messages = [
    (
        "system",
        "You are a helpful assistant that processes orders from a bookstore.",
    ),
    ("human", query),
]
result = llm.invoke(query).text
print(result)

<questions>
<question>Is Treasure Island in stock?</question>
<question>Is it possible to pay by cheque?</question>
</questions>

<name>John Smith</name>

<books>
<book>Treasure Island</book>
</books>


In [15]:
extract_by_tag(result, "question", extract_all=True)

['Is Treasure Island in stock?', 'Is it possible to pay by cheque?']

In [16]:
extract_by_tag(result, "name")

'John Smith'

In [17]:
extract_by_tag(result, "book", extract_all=True)

['Treasure Island']

## Conclusion

Entity extraction is a powerful technique using which you can extract arbitrary data using plain text descriptions.

This is particularly useful when you need to extract specific data which doesn't have clear structure. In such cases regex and other traditional extraction techniques can be very difficult to implement.

### Take aways
- Adapt this notebook to experiment with different models available through Amazon Bedrock such as Amazon Titan and AI21 Labs Jurassic models.
- Change the prompts to your specific usecase and evaluate the output of different models.
- Apply different prompt engineering principles to get better outputs. Refer to the prompt guide for your chosen model for recommendations, e.g. [here is the prompt guide for Claude](https://docs.anthropic.com/claude/docs/introduction-to-prompt-design).